# 02 — Ordinary WSI and TMA tissue extraction

RocqiPath provides two public extraction workflows:

- `run_tissue_pipeline` for ordinary resection/biopsy slides containing
  one or more contiguous tissue regions.
- `run_tma_extraction_pipeline` for circular cores and multi-stain TMA
  slides, including per-stain detection and H&E fallback.

Install `.[extraction]` and the native OpenSlide/libvips runtimes before
running the pipeline cells. Add `semantic` to the install extras and set
`detector="semantic"` to use TIAToolbox masks; Otsu remains the default.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
WSI_INPUT_DIR = DATA_ROOT / "wsi"
TMA_INPUT_DIR = DATA_ROOT / "tma"
OUTPUT_ROOT = RESULTS_ROOT

RUN_SINGLE_WSI = False
RUN_WSI_BATCH = False
RUN_TMA_BATCH = False

TARGET_MAGNIFICATION = 20.0

# Set to a float only when metadata is absent.
WSI_SOURCE_MAGNIFICATION = None
TMA_SOURCE_MAGNIFICATION = 80.0


## Discover inputs before processing

Ordinary WSI batch discovery is non-recursive. TMA discovery is
filename-based and recognizes common H&E/IHC stain tokens. Use
`target_stains` to include custom biomarkers.


In [ ]:
WSI_SUFFIXES = {
    ".svs", ".tif", ".tiff", ".ndpi", ".scn", ".mrxs", ".vms", ".vmu"
}


def direct_wsi_files(directory: Path) -> list[Path]:
    if not directory.is_dir():
        return []
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in WSI_SUFFIXES
    )


print("Ordinary WSIs:")
for path in direct_wsi_files(WSI_INPUT_DIR):
    print(" ", path.name)

print("\nTMA directory contents:")
if TMA_INPUT_DIR.is_dir():
    for path in sorted(TMA_INPUT_DIR.iterdir()):
        if path.is_file():
            print(" ", path.name)
else:
    print("  Directory does not exist yet.")


In [ ]:
from rocqipath.extraction import (
    TMAExtractionConfig,
    TissueExtractionConfig,
    extract_tissue_regions,
    run_tissue_pipeline,
    run_tma_extraction_pipeline,
)

tissue_cfg = TissueExtractionConfig(
    target_magnification=TARGET_MAGNIFICATION,
    detection_magnification=1.25,
    source_magnification=WSI_SOURCE_MAGNIFICATION,
    min_area_fraction=0.005,
    preview_scale=0.20,
    skip_existing=True,
)

tma_cfg = TMAExtractionConfig(
    target_magnification=TARGET_MAGNIFICATION,
    detection_magnification=1.25,
    source_magnification=TMA_SOURCE_MAGNIFICATION,
    min_area_fraction=0.0005,
    only_circles=True,
    min_circularity=0.60,
    per_stain_detection=True,
    fallback_to_he=True,
    box_scale=1.05,
    ihc_enhance=True,
    skip_existing=True,
)

# Optional semantic detector (requires .[extraction,semantic]):
# tissue_cfg.detector = "semantic"
# tma_cfg.detector = "semantic"
# tma_cfg.min_circularity = 0.90

print("Tissue config:", tissue_cfg.to_dict())
print("TMA config   :", tma_cfg.to_dict())


## Single ordinary WSI

Use the single-slide entry point while tuning detection. The return value is
a list of region records with relative/absolute boxes and save status.


In [ ]:
single_wsi = WSI_INPUT_DIR / "example_he.svs"

if RUN_SINGLE_WSI:
    if not single_wsi.is_file():
        raise FileNotFoundError(single_wsi)
    single_regions = extract_tissue_regions(
        str(single_wsi),
        str(OUTPUT_ROOT),
        tissue_cfg,
    )
    print(f"Detected regions: {len(single_regions)}")
    for region in single_regions:
        print(region)
else:
    print("Set RUN_SINGLE_WSI=True after editing single_wsi.")


## Batch ordinary WSIs

Each source slide writes to
`results/tissue_extraction/<slide_stem>/`. With `skip_existing=True`,
a region is skipped only when its TIFF, preview, and manifest all exist.


In [ ]:
if RUN_WSI_BATCH:
    if not WSI_INPUT_DIR.is_dir():
        raise FileNotFoundError(WSI_INPUT_DIR)
    tissue_results = run_tissue_pipeline(
        input_dir=str(WSI_INPUT_DIR),
        output_dir=str(OUTPUT_ROOT),
        cfg=tissue_cfg,
    )
    print({slide: len(regions) for slide, regions in tissue_results.items()})
else:
    print("Set RUN_WSI_BATCH=True after checking the input listing.")


## Multi-stain TMA/core extraction

H&E is the reference. With `per_stain_detection=True`, each IHC thumbnail
is detected independently. When the detected count is zero or differs from
H&E, `fallback_to_he=True` reuses the H&E relative boxes.

Custom stain labels are supported by passing them explicitly.


In [ ]:
TARGET_STAINS = ["H&E", "CD8", "CD31"]

if RUN_TMA_BATCH:
    if not TMA_INPUT_DIR.is_dir():
        raise FileNotFoundError(TMA_INPUT_DIR)
    run_tma_extraction_pipeline(
        input_dir=str(TMA_INPUT_DIR),
        output_root=str(OUTPUT_ROOT),
        cfg=tma_cfg,
        target_stains=TARGET_STAINS,
    )
else:
    print("Set RUN_TMA_BATCH=True after checking TMA filenames and stains.")


## Inspect manifests and previews

Region manifests record source/output magnification, relative and
level-0 boxes, detection source, and status. Use previews to tune
`min_area_fraction`, `min_circularity`, and `box_scale` before processing
a full cohort.


In [ ]:
import json

extraction_root = OUTPUT_ROOT / "tissue_extraction"
manifests = sorted(extraction_root.rglob("*_manifest.json")) if extraction_root.exists() else []
previews = sorted(extraction_root.rglob("*_preview.jpg")) if extraction_root.exists() else []

print(f"Manifests: {len(manifests)}")
print(f"Previews : {len(previews)}")

if manifests:
    example_manifest = manifests[0]
    payload = json.loads(example_manifest.read_text(encoding="utf-8"))
    print(f"\nExample: {example_manifest}")
    print(json.dumps(payload, indent=2)[:4000])


## Parameter tuning guide

| Symptom | Parameter to change |
|---|---|
| Small tissue fragments are missing | Lower `min_area_fraction` |
| Dust/debris becomes a region | Raise `min_area_fraction` |
| Irregular TMA cores are rejected | Lower `min_circularity` |
| Core crop clips the edge | Raise `box_scale` slightly |
| IHC core detection fails | Keep `ihc_enhance=True` and `fallback_to_he=True` |
| Output is unnecessarily large | Lower `target_magnification` |
| Plain TIFF has wrong scale | Set the correct `source_magnification` |

Do not tune every slide independently. Select representative slides,
document the chosen parameters, then apply one cohort-level config.
